# 09. Bäume: Strukturen, die ineinander stecken

Ein Rechenausdruck wie `2 + 3 * 4`, die Ordner auf deiner Festplatte, ein Stammbaum, eine Schaltung aus Widerständen, ein Mobile über dem Kinderbett:
All das sind **Bäume**. In jedem steckt etwas in etwas anderem, und das Ganze lässt sich Stück für Stück aus kleineren Teilen zusammensetzen.
In [Notebook 05](https://ktheu.github.io/bwinfN/notebooks/index.html?path=learn_bwinf1/05_breitensuche.ipynb) und [Notebook 08](https://ktheu.github.io/bwinfN/notebooks/index.html?path=learn_bwinf1/08_reihenfolgen.ipynb) hast du Graphen kennengelernt. Ein Baum ist ein Graph, bei dem es **keinen Kreis** gibt
und bei dem es von der **Wurzel** zu jedem Knoten genau einen Weg gibt. Weil jeder Teil eines Baums wieder ein (kleinerer) Baum ist, passt die **Rekursion**
([Notebook 02](https://ktheu.github.io/bwinfN/notebooks/index.html?path=learn_bwinf1/02_backtracking.ipynb) und [Notebook 10](https://ktheu.github.io/bwinfN/notebooks/index.html?path=learn_bwinf1/10_merken.ipynb)) ideal dazu.

In diesem Notebook lernst du:

- einen Baum als **verschachteltes Tupel oder verschachtelte Liste** zu schreiben (Blatt = Zahl, Knoten = Tupel oder Liste),
- Bäume **rekursiv auszuwerten**: Summe, Zahl der Blätter, Höhe, Ausgabe als Text,
- einen **Basisfall** nie zu vergessen und zu verstehen, warum eine Rekursion bei sehr tiefen Bäumen scheitert (und wie ein **Stapel** hilft),
- **Klammerfolgen** zu prüfen (mit einem **Zähler**) und mit einem **Stapel** in einen Baum zu verwandeln, und umgekehrt,
- die Laufzeit **O(Knoten)** zu begründen.

In den Wettbewerbs-Ecken prüfst du ein Mobile, suchst Schaltungen aus Widerständen und (für Schnelle) entscheidest, ob ein Baum „drehfreudig“ ist.

## 1. Ein Rechenausdruck als Baum

Der Ausdruck `2 * 3 + 4` hat eine Struktur: Zuerst wird multipliziert, dann addiert. Diese Struktur zeichnet man als Baum:

```
      +
     / \
    *   4
   / \
  2   3
```

Jeder Kreis (hier ein Zeichen oder eine Zahl) heißt **Knoten**. Der oberste, hier `+`, ist die **Wurzel**. Die Knoten direkt darunter sind seine **Kinder**.
Ein Knoten ohne Kinder (hier `2`, `3` und `4`) heißt **Blatt**. Die **Höhe** ist die Zahl der Knoten auf dem längsten Weg von der Wurzel zu einem Blatt (hier 3: `+`, `*`, `2`).

In Python schreiben wir einen solchen Baum als **verschachteltes Tupel**: Ein Blatt ist einfach die Zahl. Ein innerer Knoten ist ein Tupel `(zeichen, links, rechts)`.
Der Baum oben ist `('+', ('*', 2, 3), 4)`. Die Teile `('*', 2, 3)` und `4` sind **selbst Bäume**: der linke und der rechte **Teilbaum**.

Ob etwas ein Blatt ist, prüft man mit `isinstance(x, tuple)`: Das ist `True`, wenn `x` ein Tupel ist, sonst `False`.

#### Frage 1.1

Gegeben ist der Ausdruck `('-', ('*', 2, 5), ('+', 1, 3))`.

1. Zeichne den Baum auf Papier.
2. Wie viele Knoten hat er, wie viele davon sind Blätter, und wie hoch ist er?
3. Welchen Wert hat der Ausdruck? Rechne zuerst die Teilbäume aus.

✏️ *Deine Antwort (Doppelklick zum Bearbeiten):*

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

1. Wurzel `-` mit den Kindern `*` und `+`. Unter `*` hängen `2` und `5`, unter `+` hängen `1` und `3`.
2. **7 Knoten** (`-`, `*`, `+`, `2`, `5`, `1`, `3`), davon **4 Blätter** (`2`, `5`, `1`, `3`). Die Höhe ist **3** (zum Beispiel `-`, `*`, `2`).
3. Der linke Teilbaum ist `2 * 5 = 10`, der rechte `1 + 3 = 4`, zusammen `10 - 4 = 6`.

</details>

#### Aufgabe 1.2

Schreibe `werte_aus(baum)`. Ein Blatt ist eine Zahl, sie ist ihr eigener Wert. Bei einem Tupel `(zeichen, links, rechts)` werte zuerst beide Teilbäume **mit derselben Funktion** aus und verknüpfe die Ergebnisse mit `+`, `-` oder `*`.

**Erwartete Ausgabe:**
```
10
6
12
```

In [ ]:
ausdruck1 = ('+', ('*', 2, 3), 4)
ausdruck2 = ('-', ('*', 2, 5), ('+', 1, 3))
ausdruck3 = ('*', ('+', 1, 2), ('-', 10, ('*', 2, 3)))

def werte_aus(baum):
    # dein Code
    return 0

for ausdruck in [ausdruck1, ausdruck2, ausdruck3]:
    print(werte_aus(ausdruck))

<details>
<summary>Tipp anzeigen</summary>

Zuerst der **Basisfall**: `if not isinstance(baum, tuple): return baum`. Danach `zeichen, links, rechts = baum`, die Teilergebnisse `a = werte_aus(links)` und `b = werte_aus(rechts)` und dann je nach `zeichen` rechnen.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
ausdruck1 = ('+', ('*', 2, 3), 4)
ausdruck2 = ('-', ('*', 2, 5), ('+', 1, 3))
ausdruck3 = ('*', ('+', 1, 2), ('-', 10, ('*', 2, 3)))

def werte_aus(baum):
    if not isinstance(baum, tuple):          # Blatt: eine Zahl
        return baum
    zeichen, links, rechts = baum
    a = werte_aus(links)
    b = werte_aus(rechts)
    if zeichen == '+':
        return a + b
    if zeichen == '-':
        return a - b
    return a * b

for ausdruck in [ausdruck1, ausdruck2, ausdruck3]:
    print(werte_aus(ausdruck))
```

Das ist das Grundmuster für fast alles mit Bäumen: **Basisfall** für das Blatt, sonst zuerst die Teilbäume bearbeiten (rekursiver Aufruf) und die Ergebnisse **zusammensetzen**.
Dass die Funktion sich selbst aufruft, ist kein Problem, denn die Teilbäume sind kleiner, und irgendwann kommt man bei den Blättern an.
Der Ausdruck 3 hat den Wert `(1 + 2) * (10 - 2 * 3) = 3 * 4 = 12`.

</details>

#### Aufgabe 1.3

Schreibe `als_text(baum)`: Sie gibt den Ausdruck als Text mit **vollständiger Klammerung** zurück, zum Beispiel `((2 * 3) + 4)`. Ein Blatt ist `str(baum)`, ein Knoten ist `'('`, dann der Text des linken Teilbaums, Leerzeichen, das Zeichen, Leerzeichen, der Text des rechten Teilbaums und `')'`.

**Erwartete Ausgabe:**
```
((2 * 3) + 4)
((2 * 5) - (1 + 3))
((1 + 2) * (10 - (2 * 3)))
```

In [ ]:
ausdruck1 = ('+', ('*', 2, 3), 4)
ausdruck2 = ('-', ('*', 2, 5), ('+', 1, 3))
ausdruck3 = ('*', ('+', 1, 2), ('-', 10, ('*', 2, 3)))

def als_text(baum):
    # dein Code
    return ''

for ausdruck in [ausdruck1, ausdruck2, ausdruck3]:
    print(als_text(ausdruck))

<details>
<summary>Tipp anzeigen</summary>

Wieder: Basisfall für das Blatt (`return str(baum)`), sonst `zeichen, links, rechts = baum` und die Texte der beiden Teilbäume mit `+` zusammenfügen.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
ausdruck1 = ('+', ('*', 2, 3), 4)
ausdruck2 = ('-', ('*', 2, 5), ('+', 1, 3))
ausdruck3 = ('*', ('+', 1, 2), ('-', 10, ('*', 2, 3)))

def als_text(baum):
    if not isinstance(baum, tuple):
        return str(baum)
    zeichen, links, rechts = baum
    return '(' + als_text(links) + ' ' + zeichen + ' ' + als_text(rechts) + ')'

for ausdruck in [ausdruck1, ausdruck2, ausdruck3]:
    print(als_text(ausdruck))
```

Die Klammern im Text entstehen genau dort, wo ein innerer Knoten ist. Ein Baum lässt sich also eindeutig als Text mit Klammern schreiben. Das nutzen wir in Abschnitt 4 ganz bewusst.

</details>

#### Aufgabe 1.4

Schreibe zwei Funktionen:

- `zaehle_blaetter(baum)`: die Zahl der Blätter. Ein Blatt zählt 1, ein Knoten zählt die Blätter seiner beiden Teilbäume zusammen.
- `hoehe_ausdruck(baum)`: die Höhe. Ein Blatt hat die Höhe 1, ein Knoten die Höhe 1 plus die **größere** Höhe seiner beiden Teilbäume.

**Erwartete Ausgabe:**
```
3 Blätter, Höhe 3
4 Blätter, Höhe 3
5 Blätter, Höhe 4
```

In [ ]:
ausdruck1 = ('+', ('*', 2, 3), 4)
ausdruck2 = ('-', ('*', 2, 5), ('+', 1, 3))
ausdruck3 = ('*', ('+', 1, 2), ('-', 10, ('*', 2, 3)))

def zaehle_blaetter(baum):
    # dein Code
    return 0


def hoehe_ausdruck(baum):
    # dein Code
    return 0

for ausdruck in [ausdruck1, ausdruck2, ausdruck3]:
    print(zaehle_blaetter(ausdruck), 'Blätter, Höhe', hoehe_ausdruck(ausdruck))

<details>
<summary>Tipp anzeigen</summary>

Beide Funktionen beginnen mit dem Basisfall `if not isinstance(baum, tuple): return 1`. Bei einem Knoten rufst du sie für `links` und `rechts` auf: bei den Blättern addierst du, bei der Höhe nimmst du `max(...)` und zählst 1 für den Knoten selbst dazu.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
ausdruck1 = ('+', ('*', 2, 3), 4)
ausdruck2 = ('-', ('*', 2, 5), ('+', 1, 3))
ausdruck3 = ('*', ('+', 1, 2), ('-', 10, ('*', 2, 3)))

def zaehle_blaetter(baum):
    if not isinstance(baum, tuple):
        return 1
    zeichen, links, rechts = baum
    return zaehle_blaetter(links) + zaehle_blaetter(rechts)


def hoehe_ausdruck(baum):
    if not isinstance(baum, tuple):
        return 1
    zeichen, links, rechts = baum
    return 1 + max(hoehe_ausdruck(links), hoehe_ausdruck(rechts))

for ausdruck in [ausdruck1, ausdruck2, ausdruck3]:
    print(zaehle_blaetter(ausdruck), 'Blätter, Höhe', hoehe_ausdruck(ausdruck))
```

Beim Ausdruck 1 sind die Blätter `2`, `3`, `4` (3 Blätter, Höhe 3). Ausdruck 2 aus Frage 1.1 hat 4 Blätter und die Höhe 3, bei Ausdruck 3 sind es 5 Blätter und die Höhe 4 (der Weg `*`, `-`, `*`, `2`).

</details>

## 2. Bäume mit beliebig vielen Kindern: Ordner und Dateien

Bei Rechenausdrücken hat jeder Knoten genau zwei Kinder. Oft sind es aber **beliebig viele**: Ein Ordner enthält Dateien und andere Ordner. Wir schreiben:

- eine **Datei** als Tupel `(name, größe)` mit der Größe in KB, also einer Zahl,
- einen **Ordner** als Tupel `(name, kinder)`, wobei `kinder` eine **Liste** ist (sie darf auch leer sein).

Ob ein Tupel `k` eine Datei oder ein Ordner ist, sagt `k[1]`: Ist es eine Liste, ist `k` ein Ordner.

#### Aufgabe 2.1

Der Ordnerbaum steht unten in der Zelle (mit der Hilfsfunktion `ist_ordner`). Schreibe `gesamtgroesse(k)`: Bei einer Datei ist es ihre Größe, bei einem Ordner die **Summe** der Gesamtgrößen aller seiner Kinder.

**Erwartete Ausgabe:**
```
2569
2550
0
```

In [ ]:
ordner = ('projekt', [
    ('notizen.txt', 4),
    ('bilder', [
        ('urlaub.jpg', 900),
        ('katze.jpg', 700),
        ('alt', [('foto1.jpg', 500), ('foto2.jpg', 450)]),
    ]),
    ('code', [('spiel.py', 12), ('test.py', 3)]),
    ('leer', []),
])


def ist_ordner(k):
    return isinstance(k[1], list)

def gesamtgroesse(k):
    # dein Code
    return 0

print(gesamtgroesse(ordner))
print(gesamtgroesse(ordner[1][1]))           # der Ordner 'bilder'
print(gesamtgroesse(('leer', [])))

<details>
<summary>Tipp anzeigen</summary>

Basisfall: `if not ist_ordner(k): return k[1]`. Sonst `summe = 0`, für jedes `kind` in `k[1]` die Summe um `gesamtgroesse(kind)` erhöhen.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
ordner = ('projekt', [
    ('notizen.txt', 4),
    ('bilder', [
        ('urlaub.jpg', 900),
        ('katze.jpg', 700),
        ('alt', [('foto1.jpg', 500), ('foto2.jpg', 450)]),
    ]),
    ('code', [('spiel.py', 12), ('test.py', 3)]),
    ('leer', []),
])


def ist_ordner(k):
    return isinstance(k[1], list)

def gesamtgroesse(k):
    if not ist_ordner(k):
        return k[1]                          # Datei: ihre Größe
    summe = 0
    for kind in k[1]:
        summe = summe + gesamtgroesse(kind)
    return summe

print(gesamtgroesse(ordner))
print(gesamtgroesse(ordner[1][1]))           # der Ordner 'bilder'
print(gesamtgroesse(('leer', [])))
```

Der gesamte Ordner hat 4 + 900 + 700 + 500 + 450 + 12 + 3 = **2569** KB. Der Ordner `bilder` hat 900 + 700 + 500 + 450 = 2550, und ein leerer Ordner hat die Größe 0 (die Schleife läuft nie, die Summe bleibt 0).
Die Schleife über die Kinder ersetzt das feste „links und rechts“ aus Abschnitt 1.

</details>

#### Aufgabe 2.2

Schreibe zwei weitere Funktionen:

- `zaehle_dateien(k)`: die Zahl der **Dateien** (Blätter, die keine Ordner sind) unter `k`,
- `hoehe_ordner(k)`: die Höhe (Zahl der Knoten auf dem längsten Weg nach unten; eine Datei oder ein leerer Ordner hat die Höhe 1).

**Erwartete Ausgabe:**
```
7 Dateien
Höhe 4
Höhe des leeren Ordners 1
```

In [ ]:
ordner = ('projekt', [
    ('notizen.txt', 4),
    ('bilder', [
        ('urlaub.jpg', 900),
        ('katze.jpg', 700),
        ('alt', [('foto1.jpg', 500), ('foto2.jpg', 450)]),
    ]),
    ('code', [('spiel.py', 12), ('test.py', 3)]),
    ('leer', []),
])


def ist_ordner(k):
    return isinstance(k[1], list)

def zaehle_dateien(k):
    # dein Code
    return 0


def hoehe_ordner(k):
    # dein Code
    return 0

print(zaehle_dateien(ordner), 'Dateien')
print('Höhe', hoehe_ordner(ordner))
print('Höhe des leeren Ordners', hoehe_ordner(('leer', [])))

<details>
<summary>Tipp anzeigen</summary>

Bei `hoehe_ordner` startest du mit `hoechste = 0`, nimmst über alle Kinder das Maximum der Höhen (`max(hoechste, ...)`) und gibst `1 + hoechste` zurück. Ein leerer Ordner hat keine Kinder, die Schleife läuft nie, und die Höhe ist `1 + 0`.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
ordner = ('projekt', [
    ('notizen.txt', 4),
    ('bilder', [
        ('urlaub.jpg', 900),
        ('katze.jpg', 700),
        ('alt', [('foto1.jpg', 500), ('foto2.jpg', 450)]),
    ]),
    ('code', [('spiel.py', 12), ('test.py', 3)]),
    ('leer', []),
])


def ist_ordner(k):
    return isinstance(k[1], list)

def zaehle_dateien(k):
    if not ist_ordner(k):
        return 1
    anzahl = 0
    for kind in k[1]:
        anzahl = anzahl + zaehle_dateien(kind)
    return anzahl


def hoehe_ordner(k):
    if not ist_ordner(k):
        return 1
    hoechste = 0
    for kind in k[1]:
        hoechste = max(hoechste, hoehe_ordner(kind))
    return 1 + hoechste

print(zaehle_dateien(ordner), 'Dateien')
print('Höhe', hoehe_ordner(ordner))
print('Höhe des leeren Ordners', hoehe_ordner(('leer', [])))
```

Es gibt 7 Dateien (`notizen.txt`, `urlaub.jpg`, `katze.jpg`, `foto1.jpg`, `foto2.jpg`, `spiel.py`, `test.py`). Die Höhe ist 4: `projekt`, `bilder`, `alt`, `foto1.jpg`.
Dank des Startwerts `hoechste = 0` kommt auch ein leerer Ordner richtig heraus (siehe Abschnitt 3).

</details>

#### Aufgabe 2.3

Schreibe `zeige(k, einzug=0)`, die den Baum **eingerückt** ausgibt: Jede Ebene steht zwei Leerzeichen weiter rechts als die darüber (`'  ' * einzug`). Ein Ordner steht mit `/` und seiner Gesamtgröße da, dann folgen seine Kinder, eine Datei steht mit ihrer Größe.
`gesamtgroesse` ist schon vorgegeben.

**Erwartete Ausgabe:**
```
projekt/ (2569 KB)
  notizen.txt (4 KB)
  bilder/ (2550 KB)
    urlaub.jpg (900 KB)
    katze.jpg (700 KB)
    alt/ (950 KB)
      foto1.jpg (500 KB)
      foto2.jpg (450 KB)
  code/ (15 KB)
    spiel.py (12 KB)
    test.py (3 KB)
  leer/ (0 KB)
```

In [ ]:
ordner = ('projekt', [
    ('notizen.txt', 4),
    ('bilder', [
        ('urlaub.jpg', 900),
        ('katze.jpg', 700),
        ('alt', [('foto1.jpg', 500), ('foto2.jpg', 450)]),
    ]),
    ('code', [('spiel.py', 12), ('test.py', 3)]),
    ('leer', []),
])


def ist_ordner(k):
    return isinstance(k[1], list)

def gesamtgroesse(k):
    if not ist_ordner(k):
        return k[1]                          # Datei: ihre Größe
    summe = 0
    for kind in k[1]:
        summe = summe + gesamtgroesse(kind)
    return summe

def zeige(k, einzug=0):
    # dein Code
    pass


zeige(ordner)

<details>
<summary>Tipp anzeigen</summary>

Bei einem Ordner zuerst die Zeile mit dem Namen drucken, dann für jedes Kind `zeige(kind, einzug + 1)` aufrufen. Der Einzug wächst bei jedem Schritt nach unten um 1.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
ordner = ('projekt', [
    ('notizen.txt', 4),
    ('bilder', [
        ('urlaub.jpg', 900),
        ('katze.jpg', 700),
        ('alt', [('foto1.jpg', 500), ('foto2.jpg', 450)]),
    ]),
    ('code', [('spiel.py', 12), ('test.py', 3)]),
    ('leer', []),
])


def ist_ordner(k):
    return isinstance(k[1], list)

def gesamtgroesse(k):
    if not ist_ordner(k):
        return k[1]                          # Datei: ihre Größe
    summe = 0
    for kind in k[1]:
        summe = summe + gesamtgroesse(kind)
    return summe

def zeige(k, einzug=0):
    if ist_ordner(k):
        print('  ' * einzug + k[0] + '/ (' + str(gesamtgroesse(k)) + ' KB)')
        for kind in k[1]:
            zeige(kind, einzug + 1)
    else:
        print('  ' * einzug + k[0] + ' (' + str(k[1]) + ' KB)')

zeige(ordner)
```

Die Ausgabe zeigt die Baumstruktur auf einen Blick, und die Einrückung ist genau die Tiefe im Baum. Erst wird die Zeile eines Ordners gedruckt, dann folgen seine Kinder („Vorreihenfolge“). Würdest du die Zeile
**nach** der Schleife drucken, stünde jeder Ordner hinter seinen Kindern.
Das Ausdrucken eines Baums ist auch zum Testen sehr nützlich, wenn man ihn selbst gebaut hat.

</details>

In [ ]:
ordner = ('projekt', [
    ('notizen.txt', 4),
    ('bilder', [
        ('urlaub.jpg', 900),
        ('katze.jpg', 700),
        ('alt', [('foto1.jpg', 500), ('foto2.jpg', 450)]),
    ]),
    ('code', [('spiel.py', 12), ('test.py', 3)]),
    ('leer', []),
])


def ist_ordner(k):
    return isinstance(k[1], list)

zaehler = [0]                                # eine Liste, damit die Funktion den Wert verändern darf


def gesamtgroesse_gezaehlt(k):
    zaehler[0] = zaehler[0] + 1
    if not ist_ordner(k):
        return k[1]
    summe = 0
    for kind in k[1]:
        summe = summe + gesamtgroesse_gezaehlt(kind)
    return summe


print(gesamtgroesse_gezaehlt(ordner), 'KB,', zaehler[0], 'Aufrufe')

#### Frage 2.4

Die Zelle oben zählt, wie oft `gesamtgroesse_gezaehlt` aufgerufen wird (Der Ordnerbaum steht dazu noch einmal in der Zelle).

1. Wie viele **Knoten** hat der Ordnerbaum (Dateien und Ordner)? Vergleiche mit der Zahl der Aufrufe.
2. Wie lässt sich die Laufzeit in der O-Notation mit der Zahl der Knoten `n` ausdrücken? Was passiert bei doppelt so vielen Knoten?
3. Ein Student schreibt `gesamtgroesse` so, dass sie für jedes Kind **zweimal** `gesamtgroesse(kind)` aufruft („zur Sicherheit“). Wie viele Aufrufe gibt es dann bei einer Kette von Ordnern, die 30-mal ineinander liegen?

✏️ *Deine Antwort (Doppelklick zum Bearbeiten):*

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

1. Es gibt 7 Dateien und 5 Ordner (`projekt`, `bilder`, `alt`, `code`, `leer`), zusammen **12 Knoten**. Die Funktion wird genau 12-mal aufgerufen: **einmal pro Knoten**.
2. Jeder Knoten wird genau einmal besucht, die Arbeit pro Knoten ist klein (ein Aufruf und eine Addition pro Kind). Die Laufzeit ist **O(n)**. Bei doppelt so vielen Knoten dauert es etwa doppelt so lang.
3. Auf jeder Ebene verdoppelt sich die Zahl der Aufrufe: Es sind etwa 2³⁰ ≈ 1 Milliarde Aufrufe. Das ist die Gefahr, wenn man einen Teilbaum **mehrfach** bearbeitet. Bäume sind schnell, wenn jeder Knoten **einmal** drankommt.

</details>

## 3. Finde den Fehler

**Fehler 1: kein Basisfall.** Dieses Programm soll `2 + 3` ausrechnen:

```python
def werte_aus_falsch(baum):
    zeichen, links, rechts = baum
    a = werte_aus_falsch(links)
    b = werte_aus_falsch(rechts)
    if zeichen == '+':
        return a + b
    return a * b

print(werte_aus_falsch(('+', 2, 3)))
```

Fehlermeldung: `TypeError: cannot unpack non-iterable int object`

Die Funktion packt `baum` aus, aber bei `2` (einem Blatt) gibt es nichts auszupacken. Es fehlt der **Basisfall** am Anfang: `if not isinstance(baum, tuple): return baum`.
Ohne ihn gibt es keinen Punkt, an dem die Rekursion aufhört. Hier merkt man es sofort an einer Fehlermeldung, bei anderen Bäumen (zum Beispiel einer Rekursion, die immer wieder denselben Knoten aufruft) läuft das Programm so lange, bis Python mit einem `RecursionError` abbricht.

**Fehler 2: der Knoten ohne Kinder.** Diese Höhenberechnung für den Ordnerbaum aus Abschnitt 2 stürzt bei einem **leeren Ordner** ab:

```python
def hoehe_falsch(k):
    if not isinstance(k[1], list):
        return 1
    return 1 + max([hoehe_falsch(kind) for kind in k[1]])


print(hoehe_falsch(('leer', [])))
```

Fehlermeldung: `ValueError: max() iterable argument is empty`

Die Liste der Kinder ist leer, und `max` einer leeren Liste ist nicht definiert. In `hoehe_ordner` aus Aufgabe 2.2 gibt es das Problem nicht, dort steht der Startwert `hoechste = 0`.
Wenn man mit `max` oder `min` über die Kinder arbeitet, muss man klären, was bei **0 Kindern** herauskommen soll. Prüfe jeden Baumcode mit dem **kleinsten** Baum (ein einziger Knoten) und mit einem Knoten **ohne** Kinder.

## 4. Klammerfolgen

Bäume lassen sich sehr kurz als **Folge von Klammern** aufschreiben: Jeder Knoten ist ein Klammerpaar, und zwischen den Klammern stehen seine Kinder. Ein Blatt ist `()`. Ein Knoten mit zwei Blättern als Kindern ist `(()())`.
Der Baum `(()(()))` besteht aus einer Wurzel mit zwei Kindern: einem Blatt und einem Knoten mit einem Kind.

Die Folge ist nur dann **gültig**, wenn jede öffnende Klammer eine passende schließende hat und keine schließende zu früh kommt. Man prüft das mit einem **Zähler**: Bei `(` zählt man 1 hoch, bei `)` 1 runter.
Rutscht der Zähler unter 0, kam eine Klammer zu früh. Ist er am Ende nicht 0, bleibt eine Klammer offen.

#### Frage 4.1

Welche dieser Klammerfolgen sind gültig, welche nicht? Warum?

`(()())`, `()()`, `(()`, `())(`, `)(`

✏️ *Deine Antwort (Doppelklick zum Bearbeiten):*

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

- `(()())` ist **gültig** (Wurzel mit zwei Blättern).
- `()()` ist gültig, aber **kein einzelner Baum**: Es sind zwei Bäume nebeneinander (ein „Wald“). Für einen Baum muss ein einziges Klammerpaar alles umschließen.
- `(()` ist **ungültig**: Eine öffnende Klammer bleibt offen (Zähler am Ende 1).
- `())(` ist **ungültig**: Nach `())` ist der Zähler schon −1, die dritte Klammer kam zu früh. Das ist ein Fall, in dem es gleich viele `(` wie `)` gibt, die Folge aber trotzdem falsch ist.
- `)(` ist **ungültig**: Der Zähler wird sofort −1.

</details>

#### Aufgabe 4.2

Schreibe `ist_gueltig(text)`: Sie gibt `True` zurück, wenn die Klammerfolge gültig ist, sonst `False`. Gehe die Zeichen durch, zähle bei `(` hoch und bei `)` runter, und gib sofort `False` zurück, wenn der Zähler unter 0 fällt. Am Ende muss der Zähler genau 0 sein.

**Erwartete Ausgabe:**
```
(()()) True
()() True
(() False
())( False
)( False
```

In [ ]:
def ist_gueltig(text):
    # dein Code
    return False

for text in ['(()())', '()()', '(()', '())(', ')(']:
    print(text, ist_gueltig(text))

<details>
<summary>Tipp anzeigen</summary>

`zaehler = 0`, dann `for zeichen in text:`. Bei `'('` den Zähler um 1 erhöhen, sonst um 1 senken und prüfen, ob er kleiner als 0 ist. Nach der Schleife `return zaehler == 0`.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
def ist_gueltig(text):
    zaehler = 0
    for zeichen in text:
        if zeichen == '(':
            zaehler = zaehler + 1
        else:
            zaehler = zaehler - 1
            if zaehler < 0:
                return False             # schließende Klammer zu früh
    return zaehler == 0

for text in ['(()())', '()()', '(()', '())(', ')(']:
    print(text, ist_gueltig(text))
```

Die Schleife läuft einmal über den Text: **O(n)** bei n Zeichen. Weil der Zähler schon beim ersten zu frühen `)` abbricht, erkennt die Funktion `())(` und `)(` richtig.

</details>

#### Aufgabe 4.3: Was wird ausgegeben?

Überlege zuerst, was der Code ausgibt, und schreibe es auf. Führe die Zelle erst danach aus.

In [ ]:
def ist_gueltig_falsch(text):
    return text.count('(') == text.count(')')


for text in ['(()', '())(', ')(', '(())']:
    print(text, ist_gueltig_falsch(text))

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

**Ausgabe:**
```
(() False
())( True
)( True
(()) True
```

Ausgegeben wird `False`, `True`, `True`, `True`. Die Funktion zählt nur, ob es **gleich viele** öffnende und schließende Klammern gibt. Bei `())(` und `)(` stimmt die Zahl, trotzdem sind beide Folgen ungültig, denn eine schließende Klammer kommt **vor** der öffnenden.
Es kommt also auf die **Reihenfolge** an, deshalb braucht man den Zähler, der nie unter 0 fallen darf.

</details>

### Von der Klammerfolge zum Baum

Wie machen wir aus `(()(()))` einen Baum im Speicher? Wir nehmen **verschachtelte Listen**: Ein Knoten ist die **Liste seiner Kinder**, ein Blatt ist die leere Liste `[]`. Der Baum `(()(()))` wird zu `[[], [[]]]`.

Dazu brauchen wir einen **Stapel** (*Stack*, eine Liste, bei der man nur hinten etwas anfügt und wegnimmt, wie in learn_jwinf3, Notebook 21): Er enthält die Knoten, deren Klammer **gerade noch offen** ist.

- Bei `(` legen wir einen neuen leeren Knoten `[]` oben auf den Stapel.
- Bei `)` nehmen wir den obersten Knoten vom Stapel. Er ist fertig und wird als **Kind** an den Knoten darunter gehängt.

Damit unten immer ein Knoten da ist, an den die Wurzel gehängt werden kann, beginnen wir mit einem Stapel, der schon einen leeren Hilfsknoten enthält. Am Ende steht die Wurzel als einziges Kind des Hilfsknotens.

#### Aufgabe 4.4

Schreibe `klammern_zu_baum(text)` mit einem Stapel wie oben beschrieben. Du darfst davon ausgehen, dass `text` gültig ist und einen einzigen Baum beschreibt. Die Funktion gibt die Wurzel als verschachtelte Liste zurück.

**Erwartete Ausgabe:**
```
[]
[[], []]
[[], [[]]]
```

In [ ]:
def klammern_zu_baum(text):
    stapel = [[]]
    # dein Code
    return stapel[0]

print(klammern_zu_baum('()'))
print(klammern_zu_baum('(()())'))
print(klammern_zu_baum('(()(()))'))

<details>
<summary>Tipp anzeigen</summary>

Bei `'('`: `stapel.append([])`. Sonst: `knoten = stapel.pop()` und `stapel[-1].append(knoten)`. `stapel[-1]` ist das oberste Element des Stapels.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
def klammern_zu_baum(text):
    stapel = [[]]                               # ein Hilfsknoten ganz unten
    for zeichen in text:
        if zeichen == '(':
            stapel.append([])                   # neuer Knoten, seine Klammer ist offen
        else:
            knoten = stapel.pop()               # fertig ...
            stapel[-1].append(knoten)           # ... und als Kind an den Knoten darunter hängen
    return stapel[0][0]                         # die Wurzel

print(klammern_zu_baum('()'))
print(klammern_zu_baum('(()())'))
print(klammern_zu_baum('(()(()))'))
```

Das einzelne Paar `()` ist ein Blatt, also `[]`. `(()())` ist `[[], []]`: eine Wurzel mit zwei Blättern. `(()(()))` ist `[[], [[]]]`.
Der Stapel hat immer genau so viele Elemente, wie Klammern offen sind (plus der Hilfsknoten): Der Zähler aus Aufgabe 4.2 ist die Länge des Stapels minus 1.
Jede Klammer wird einmal angesehen: **O(n)**.

</details>

#### Aufgabe 4.5

Schreibe drei Funktionen. Der Baum ist eine verschachtelte Liste wie in 4.4.

- `baum_zu_klammern(baum)`: der Baum als Klammerfolge, **rekursiv**: `'('`, dann die Klammerfolgen aller Kinder hintereinander, dann `')'`.
- `hoehe_baum(baum)`: die Höhe (ein Blatt hat die Höhe 1), rekursiv wie `hoehe_ordner` in 2.2.
- `max_tiefe(text)`: die Höhe **direkt aus der Klammerfolge**, ohne Baum: der größte Wert, den der Zähler aus 4.2 annimmt.

Die Probe: Hin und zurück muss wieder den Text ergeben, und `hoehe_baum` und `max_tiefe` müssen übereinstimmen.
`klammern_zu_baum` ist schon vorgegeben.

**Erwartete Ausgabe:**
```
() True 1 1
(()()) True 2 2
(()(())) True 3 3
((()())(()()())) True 3 3
```

In [ ]:
def klammern_zu_baum(text):
    stapel = [[]]                               # ein Hilfsknoten ganz unten
    for zeichen in text:
        if zeichen == '(':
            stapel.append([])                   # neuer Knoten, seine Klammer ist offen
        else:
            knoten = stapel.pop()               # fertig ...
            stapel[-1].append(knoten)           # ... und als Kind an den Knoten darunter hängen
    return stapel[0][0]                         # die Wurzel

def baum_zu_klammern(baum):
    # dein Code
    return ''


def hoehe_baum(baum):
    # dein Code
    return 0


def max_tiefe(text):
    # dein Code
    return 0

for text in ['()', '(()())', '(()(()))', '((()())(()()()))']:
    baum = klammern_zu_baum(text)
    print(text, baum_zu_klammern(baum) == text, hoehe_baum(baum), max_tiefe(text))

<details>
<summary>Tipp anzeigen</summary>

`baum_zu_klammern`: `text = '('`, für jedes `kind` in `baum` den Text von `baum_zu_klammern(kind)` anhängen, am Ende `')'` anfügen. Bei einem Blatt (leere Liste) läuft die Schleife nie, es bleibt `()`.
`max_tiefe`: wie in 4.2, nur dass du nach jedem `(` den bisher größten Wert merkst.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
def klammern_zu_baum(text):
    stapel = [[]]                               # ein Hilfsknoten ganz unten
    for zeichen in text:
        if zeichen == '(':
            stapel.append([])                   # neuer Knoten, seine Klammer ist offen
        else:
            knoten = stapel.pop()               # fertig ...
            stapel[-1].append(knoten)           # ... und als Kind an den Knoten darunter hängen
    return stapel[0][0]                         # die Wurzel

def baum_zu_klammern(baum):
    text = '('
    for kind in baum:
        text = text + baum_zu_klammern(kind)
    return text + ')'


def hoehe_baum(baum):
    hoechste = 0
    for kind in baum:
        hoechste = max(hoechste, hoehe_baum(kind))
    return 1 + hoechste


def max_tiefe(text):
    tiefe = 0
    groesste = 0
    for zeichen in text:
        if zeichen == '(':
            tiefe = tiefe + 1
            groesste = max(groesste, tiefe)
        else:
            tiefe = tiefe - 1
    return groesste

for text in ['()', '(()())', '(()(()))', '((()())(()()()))']:
    baum = klammern_zu_baum(text)
    print(text, baum_zu_klammern(baum) == text, hoehe_baum(baum), max_tiefe(text))
```

Die Probe stimmt in jeder Zeile: Der Text ist nach dem Hin und Zurück wieder derselbe (`True`), und die Höhe aus dem Baum ist gleich dem größten Zählerstand im Text. Der letzte Baum `((()())(()()()))` hat die Höhe 3 (Wurzel, Kind, Blatt).
Ein Baum und seine Klammerfolge enthalten **dieselbe Information**: Man kann sie ineinander umwandeln, ohne etwas zu verlieren. Für das Programmieren ist der Baum meist bequemer, die Klammerfolge ist praktisch zum Speichern und Eingeben.

</details>

#### Frage 4.6

1. Ein Baum hat 12 Knoten. Wie viele **Kanten** (Verbindungen zwischen Knoten) hat er, und warum? Wie viele Zeichen hat seine Klammerfolge?
2. Warum gibt es in einem Baum keinen **Kreis** (wie bei Notebook 08)? Denke daran, wie viele Elternknoten ein Knoten hat.
3. Wie viele Schritte brauchen `klammern_zu_baum`, `baum_zu_klammern` und `hoehe_baum` für einen Baum mit n Knoten (O-Notation)? Warum ist das nicht O(n²)?

✏️ *Deine Antwort (Doppelklick zum Bearbeiten):*

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

1. Jeder Knoten außer der Wurzel hat genau **einen Elternknoten** und damit genau eine Kante nach oben: **11 Kanten**. Die Klammerfolge hat zwei Zeichen je Knoten, also **24 Zeichen**.
2. Ein Kreis bräuchte einen Knoten mit zwei Wegen von der Wurzel her, also zwei Elternknoten, oder einen Knoten, der sein eigener Vorfahr ist. Bei einem Baum gibt es zu jedem Knoten **genau einen** Weg von der Wurzel, deshalb keinen Kreis.
3. Alle drei brauchen **O(n)**: Jeder Knoten wird einmal bearbeitet (bei `klammern_zu_baum` jedes der 2n Zeichen einmal), mit wenigen einfachen Rechenschritten je Knoten. Würde man in jedem Knoten **den ganzen Teilbaum noch einmal** ansehen (oder wie in Frage 2.4 zweimal aufrufen), könnte es mehr werden.

</details>

## 5. Wenn der Baum sehr tief ist: Rekursionsgrenze

In den Notebooks 08 und 10 hast du gesehen, dass Python eine Rekursion nach etwa **1 000** ineinander liegenden Aufrufen mit einem `RecursionError` abbricht. Bei Bäumen heißt das: Ist der Baum **höher** als etwa 1 000, scheitert die rekursive Höhenberechnung, auch wenn der Baum klein genug für den Speicher ist.
Ein Beispiel ist eine **Kette**: ein Knoten, der ein Kind hat, das ein Kind hat, und so weiter.

#### Aufgabe 5.1: Was wird ausgegeben?

Überlege zuerst, was der Code ausgibt, und schreibe es auf. Führe die Zelle erst danach aus.

In [ ]:
def hoehe_baum(baum):
    hoechste = 0
    for kind in baum:
        hoechste = max(hoechste, hoehe_baum(kind))
    return 1 + hoechste


kette = []
for i in range(3000):
    kette = [kette]               # ein neuer Knoten über der bisherigen Kette

try:
    print(hoehe_baum(kette))
except RecursionError:
    print('zu tief für die Rekursion')

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

**Ausgabe:**
```
zu tief für die Rekursion
```

Ausgegeben wird `zu tief für die Rekursion`. Die Kette hat 3 001 Knoten übereinander, die Funktion ruft sich 3 001-mal ineinander auf, und das ist mehr, als Python erlaubt.
Die Rekursion merkt sich bei jedem Aufruf, wo sie gerade war, und das braucht Speicher, deshalb die Grenze. Die Lösung: Man verwaltet die **noch zu bearbeitenden Knoten selbst** in einem Stapel (wie in 4.4), dann gibt es keine ineinander liegenden Aufrufe.

</details>

#### Aufgabe 5.2

Schreibe `hoehe_stapel(baum)` **ohne Rekursion**. Der Stapel enthält Paare `(knoten, tiefe)` der noch nicht bearbeiteten Knoten, zu Beginn `[(baum, 1)]`. Solange der Stapel nicht leer ist: Nimm das oberste Paar herunter, merke dir die größte Tiefe, und lege alle Kinder mit der Tiefe `tiefe + 1` auf den Stapel.

**Erwartete Ausgabe:**
```
3001
3
```

In [ ]:
def hoehe_stapel(baum):
    stapel = [(baum, 1)]
    groesste = 0
    # dein Code
    return groesste

kette = []
for i in range(3000):
    kette = [kette]
print(hoehe_stapel(kette))
print(hoehe_stapel([[], [[]]]))

<details>
<summary>Tipp anzeigen</summary>

`while len(stapel) > 0:`, dann `knoten, tiefe = stapel.pop()`, `groesste = max(groesste, tiefe)` und `for kind in knoten: stapel.append((kind, tiefe + 1))`.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
def hoehe_stapel(baum):
    stapel = [(baum, 1)]                         # (Knoten, Tiefe), die noch zu bearbeiten sind
    groesste = 0
    while len(stapel) > 0:
        knoten, tiefe = stapel.pop()
        groesste = max(groesste, tiefe)
        for kind in knoten:
            stapel.append((kind, tiefe + 1))
    return groesste

kette = []
for i in range(3000):
    kette = [kette]
print(hoehe_stapel(kette))
print(hoehe_stapel([[], [[]]]))
```

Die Kette hat die Höhe 3 001, der kleine Baum `[[], [[]]]` die Höhe 3. Jeder Knoten kommt genau einmal auf den Stapel und wird einmal heruntergenommen: wieder **O(n)**.
Die Reihenfolge, in der die Knoten bearbeitet werden, ist eine andere als bei der Rekursion, aber für die Höhe ist sie egal. Diese Methode nennt man **Tiefensuche mit Stapel**, du kennst sie aus learn_jwinf3 (Notebook 21).
Bei den Wettbewerbsaufgaben dieses Lehrgangs sind die Bäume meist so klein, dass die Rekursion genügt. Wenn eine Aufgabe Bäume mit Hunderttausenden von Knoten hat, nimm den Stapel.

</details>

## Wettbewerbs-Ecke: Mobile

*Angelehnt an Aufgabe 2 „Mobile“ der 1. Runde des 33. Bundeswettbewerbs Informatik (2014/15).*

Ein Mobile besteht aus waagerechten **Balken**, an denen **Figuren** oder weitere Balken hängen. Gewichte von Balken und Drähten zählen nicht, nur die Figuren haben ein Gewicht.
Jede Struktur (Figur oder tiefer hängender Balken) hängt an einer Stelle mit einem **Abstand** vom Aufhängepunkt des Balkens: links negativ, rechts positiv.
Ein Balken ist **balanciert**, wenn die Summe aus **Gewicht mal Abstand** aller seiner Strukturen 0 ist. Das Mobile ist balanciert, wenn **alle** seine Balken balanciert sind.
Das **Gewicht eines Balkens** ist die Summe der Gewichte seiner Strukturen.

Wir speichern ein Mobile in einer Textdatei, **ein Balken pro Zeile**: Die Zeile besteht aus Zahlenpaaren `w a` von links nach rechts. Eine Figur hat das Gewicht `w` (größer als 0) und den Abstand `a`.
Hängt an dieser Stelle ein **Balken**, steht `w = 0`. Welcher Balken gemeint ist, ergibt sich aus der Reihenfolge: Direkt nach der Zeile eines Balkens kommen die Zeilen seiner Unterbalken, einer nach dem anderen, jeweils mit allen ihren Unterbalken.

Beispiel `daten/mobile1.txt` (das Mobile aus der Aufgabe):

```
0 -4 2 -1 3 2 2 4
1 -2 2 1
```

Die erste Zeile: ein Unterbalken (`0`) im Abstand −4, eine Figur mit Gewicht 2 im Abstand −1, eine Figur 3 im Abstand 2 und eine Figur 2 im Abstand 4. Die zweite Zeile ist der Unterbalken: Figur 1 im Abstand −2 und Figur 2 im Abstand 1.
Seine Rechnung: 1 · (−2) + 2 · 1 = 0, er wiegt 3. Der obere Balken: 3 · (−4) + 2 · (−1) + 3 · 2 + 2 · 4 = −12 − 2 + 6 + 8 = 0.

#### Aufgabe E.1

Die Funktionen `lies_zeilen` und `baue_balken` sind vorgegeben. `baue_balken` ist rekursiv: Trifft sie auf `w = 0`, ruft sie sich für die **nächste freie Zeile** auf und bekommt den Unterbalken samt der Nummer der Zeile zurück, mit der es weitergeht.
Führe die Zelle aus und lies das Ergebnis: Ein Balken ist eine Liste von Paaren `(struktur, abstand)`.

**Erwartete Ausgabe:**
```
[([(1, -2), (2, 1)], -4), (2, -1), (3, 2), (2, 4)]
Zeilen gelesen: 2 von 2
```

In [ ]:
def lies_zeilen(dateiname):
    with open(dateiname, encoding='utf-8') as datei:
        return datei.read().splitlines()


def baue_balken(zeilen, z):
    """Baut den Balken aus Zeile z samt allen Unterbalken.
    Ein Balken ist eine Liste von Paaren (struktur, abstand); struktur ist eine Zahl (Figur) oder wieder ein Balken (Liste).
    Rückgabe: (balken, Nummer der nächsten noch nicht gelesenen Zeile)."""
    zahlen = []
    for wort in zeilen[z].split():
        zahlen.append(int(wort))
    naechste = z + 1
    balken = []
    for i in range(0, len(zahlen), 2):
        w = zahlen[i]
        a = zahlen[i + 1]
        if w == 0:                                    # hier hängt ein Unterbalken
            struktur, naechste = baue_balken(zeilen, naechste)
        else:
            struktur = w                              # eine Figur
        balken.append((struktur, a))
    return balken, naechste

zeilen = lies_zeilen('daten/mobile1.txt')
mobile1, naechste = baue_balken(zeilen, 0)
print(mobile1)
print('Zeilen gelesen:', naechste, 'von', len(zeilen))

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
def lies_zeilen(dateiname):
    with open(dateiname, encoding='utf-8') as datei:
        return datei.read().splitlines()


def baue_balken(zeilen, z):
    """Baut den Balken aus Zeile z samt allen Unterbalken.
    Ein Balken ist eine Liste von Paaren (struktur, abstand); struktur ist eine Zahl (Figur) oder wieder ein Balken (Liste).
    Rückgabe: (balken, Nummer der nächsten noch nicht gelesenen Zeile)."""
    zahlen = []
    for wort in zeilen[z].split():
        zahlen.append(int(wort))
    naechste = z + 1
    balken = []
    for i in range(0, len(zahlen), 2):
        w = zahlen[i]
        a = zahlen[i + 1]
        if w == 0:                                    # hier hängt ein Unterbalken
            struktur, naechste = baue_balken(zeilen, naechste)
        else:
            struktur = w                              # eine Figur
        balken.append((struktur, a))
    return balken, naechste

zeilen = lies_zeilen('daten/mobile1.txt')
mobile1, naechste = baue_balken(zeilen, 0)
print(mobile1)
print('Zeilen gelesen:', naechste, 'von', len(zeilen))
```

Der obere Balken hat vier Strukturen: den Unterbalken `[(1, -2), (2, 1)]` im Abstand −4, danach die Figuren 2, 3 und 2 in den Abständen −1, 2 und 4.
Beide Zeilen der Datei wurden gelesen (2 von 2). Genau das prüft man später, um zu erkennen, dass eine Datei nicht überzählige Zeilen enthält.

</details>

#### Aufgabe E.2

Schreibe zwei rekursive Funktionen:

- `gewicht(struktur)`: Ein **Balken** ist eine Liste von Paaren `(teil, abstand)`. Eine **Figur** ist eine Zahl, sie ist ihr eigenes Gewicht. Das Gewicht eines Balkens ist die Summe der Gewichte seiner Teile.
- `ist_balanciert(balken)`: `True`, wenn die Summe von `gewicht(teil) * abstand` den Wert 0 ergibt **und** alle Unterbalken balanciert sind.

Die Dateien `mobile2.txt` (balanciert) und `mobile3.txt` (die letzte Zeile ist verändert) kommen dazu.

**Erwartete Ausgabe:**
```
mobile1 Gewicht 10 balanciert: True
mobile2 Gewicht 8 balanciert: True
mobile3 Gewicht 9 balanciert: False
```

In [ ]:
def lies_zeilen(dateiname):
    with open(dateiname, encoding='utf-8') as datei:
        return datei.read().splitlines()


def baue_balken(zeilen, z):
    """Baut den Balken aus Zeile z samt allen Unterbalken.
    Ein Balken ist eine Liste von Paaren (struktur, abstand); struktur ist eine Zahl (Figur) oder wieder ein Balken (Liste).
    Rückgabe: (balken, Nummer der nächsten noch nicht gelesenen Zeile)."""
    zahlen = []
    for wort in zeilen[z].split():
        zahlen.append(int(wort))
    naechste = z + 1
    balken = []
    for i in range(0, len(zahlen), 2):
        w = zahlen[i]
        a = zahlen[i + 1]
        if w == 0:                                    # hier hängt ein Unterbalken
            struktur, naechste = baue_balken(zeilen, naechste)
        else:
            struktur = w                              # eine Figur
        balken.append((struktur, a))
    return balken, naechste

def gewicht(struktur):
    # dein Code
    return 0


def ist_balanciert(balken):
    # dein Code
    return False

for nr in [1, 2, 3]:
    zeilen = lies_zeilen('daten/mobile' + str(nr) + '.txt')
    mobile, naechste = baue_balken(zeilen, 0)
    print('mobile' + str(nr), 'Gewicht', gewicht(mobile), 'balanciert:', ist_balanciert(mobile))

<details>
<summary>Tipp anzeigen</summary>

Eine Figur erkennst du daran, dass sie **keine Liste** ist (`isinstance(struktur, list)`). Bei `ist_balanciert` gehst du in einer Schleife über alle `(teil, abstand)`: Ist `teil` eine Liste, muss auch `ist_balanciert(teil)` stimmen, sonst gleich `False`. Dann das Moment `gewicht(teil) * abstand` aufsummieren. Am Ende ist die Summe 0 oder nicht.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
def lies_zeilen(dateiname):
    with open(dateiname, encoding='utf-8') as datei:
        return datei.read().splitlines()


def baue_balken(zeilen, z):
    """Baut den Balken aus Zeile z samt allen Unterbalken.
    Ein Balken ist eine Liste von Paaren (struktur, abstand); struktur ist eine Zahl (Figur) oder wieder ein Balken (Liste).
    Rückgabe: (balken, Nummer der nächsten noch nicht gelesenen Zeile)."""
    zahlen = []
    for wort in zeilen[z].split():
        zahlen.append(int(wort))
    naechste = z + 1
    balken = []
    for i in range(0, len(zahlen), 2):
        w = zahlen[i]
        a = zahlen[i + 1]
        if w == 0:                                    # hier hängt ein Unterbalken
            struktur, naechste = baue_balken(zeilen, naechste)
        else:
            struktur = w                              # eine Figur
        balken.append((struktur, a))
    return balken, naechste

def gewicht(struktur):
    if not isinstance(struktur, list):               # eine Figur
        return struktur
    summe = 0
    for teil, abstand in struktur:
        summe = summe + gewicht(teil)
    return summe


def ist_balanciert(balken):
    moment = 0
    for teil, abstand in balken:
        if isinstance(teil, list):
            if not ist_balanciert(teil):             # ein Unterbalken ist schief
                return False
        moment = moment + gewicht(teil) * abstand
    return moment == 0

for nr in [1, 2, 3]:
    zeilen = lies_zeilen('daten/mobile' + str(nr) + '.txt')
    mobile, naechste = baue_balken(zeilen, 0)
    print('mobile' + str(nr), 'Gewicht', gewicht(mobile), 'balanciert:', ist_balanciert(mobile))
```

`mobile1` wiegt 3 + 2 + 3 + 2 = **10** und ist balanciert. `mobile2` hat drei Balken und wiegt 3 + 2 + 3 = **8**, auch balanciert (Rechnung: Der obere Balken hat 3 · (−2) + 2 · (−3) + 3 · 4 = 0).
Bei `mobile3` ist die letzte Zeile `1 -1 2 1` statt `1 -1 1 1`: Dieser Unterbalken wiegt 3 und hat das Moment 1 · (−1) + 2 · 1 = 1, ist also schief. Er wird zuerst geprüft (`False`), und der obere Balken wäre mit 3 · (−2) + 3 · (−3) + 3 · 4 = −3 ebenfalls schief.
**Gewicht und Prüfung** sind dieselbe Art von Rekursion wie bei den Ordnern: Die Antwort für einen Balken setzt sich aus den Antworten für die Teilbalken zusammen.

</details>

#### Frage E.3

1. `ist_balanciert` ruft für jeden Teil `gewicht(teil)` auf, und `gewicht` geht durch den **ganzen** Teilbaum. Wie oft wird ein Knoten ungefähr angesehen, wenn das Mobile die Höhe `h` hat? Ist das für Mobiles mit höchstens 4 Strukturen je Balken und wenigen Dutzend Figuren ein Problem?
2. Die echte Aufgabe ist schwerer: Das Programm soll aus den Gewichten von n Figuren ein **balanciertes Mobile bauen**, mit höchstens 4 Strukturen je Balken. Welche Teile aus diesem Notebook helfen dir dabei? Was müsstest du für einen Balken ausrechnen?

✏️ *Deine Antwort (Doppelklick zum Bearbeiten):*

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

1. Jeder Knoten wird ungefähr **einmal pro Ebene** über ihm angesehen (bei jedem Balken über ihm wird `gewicht` neu gerechnet), also höchstens `h`-mal. Das sind insgesamt **O(n · h)** Schritte. Bei wenigen Dutzend Figuren ist das winzig. Wer es genau machen will, lässt `ist_balanciert` das Gewicht gleich mit zurückgeben, dann bleibt es O(n).
2. Das Mobile ist ein Baum, und die Teile aus diesem Notebook (Baum als verschachtelte Liste, Gewicht rekursiv, Ausgabe als Text, Prüfung `ist_balanciert`) helfen beim **Prüfen** des fertigen Mobiles. Fürs **Bauen** muss man für einen Balken aus den Gewichten der Teile Abstände finden, deren gewichtete Summe 0 ist, und dann die schwereren oder leichteren Teile geschickt zu Balken zusammenfassen. Wie das geht, zeigt das Lösungsnotebook zur Aufgabe (Link am Ende).

</details>

## Wettbewerbs-Ecke: Widerstand

*Angelehnt an Aufgabe 5 „Widerstand“ der 1. Runde des 37. Bundeswettbewerbs Informatik (2018/19).*

Zelda hat in ihrer Grabbelkiste Widerstände mit verschiedenen Werten (in Ohm, Ω). Schaltet man zwei Widerstände `a` und `b`

- **in Reihe**, ist der Gesamtwiderstand `a + b`,
- **parallel**, ist er `a · b / (a + b)`.

Mit mehreren Widerständen kann man das verschachteln. Eine Schaltung ist ein **Baum**: Die Blätter sind Widerstände aus der Kiste, jeder innere Knoten ist `('+', links, rechts)` (Reihe) oder `('|', links, rechts)` (parallel).
Zum Beispiel ist `('|', ('+', 180, 820), 1000)` die Reihenschaltung von 180 Ω und 820 Ω (zusammen 1 000 Ω), parallel zu einem Widerstand von 1 000 Ω: Das ergibt 500 Ω.

Gesucht ist zu einem Wunschwert `R` die Schaltung aus **höchstens k** verschiedenen Widerständen der Kiste, deren Wert **am nächsten** an `R` liegt. In der Datei `daten/widerstaende.txt` stehen 24 Widerstandswerte, ein Wert pro Zeile.
Hier genügt `k` bis 3 (die Aufgabe geht bis 4, das braucht aber mehr Rechenzeit).

#### Aufgabe E.4

Schreibe `schaltungswert(schaltung)` (der Gesamtwiderstand, rekursiv wie `werte_aus`) und `als_formel(schaltung)` (die Schaltung als Text mit Klammern, `+` für Reihe und `|` für parallel, wie `als_text` aus 1.3).

**Erwartete Ausgabe:**
```
((180 + 820) | 1000) = 500.0 Ω
(((1200 | 3300) + 120) | 1000) = 500.0 Ω
```

In [ ]:
def schaltungswert(schaltung):
    # dein Code
    return 0


def als_formel(schaltung):
    # dein Code
    return ''

probe = ('|', ('+', 180, 820), 1000)
print(als_formel(probe), '=', schaltungswert(probe), 'Ω')
probe2 = ('|', ('+', ('|', 1200, 3300), 120), 1000)
print(als_formel(probe2), '=', round(schaltungswert(probe2), 3), 'Ω')

<details>
<summary>Tipp anzeigen</summary>

Beide beginnen mit dem Basisfall für einen einzelnen Widerstand (`not isinstance(schaltung, tuple)`). Sonst `zeichen, links, rechts = schaltung`. Beim Wert rechnest du je nach `zeichen` `a + b` oder `a * b / (a + b)`.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
def schaltungswert(schaltung):
    if not isinstance(schaltung, tuple):             # ein einzelner Widerstand
        return schaltung
    zeichen, links, rechts = schaltung
    a = schaltungswert(links)
    b = schaltungswert(rechts)
    if zeichen == '+':
        return a + b
    return a * b / (a + b)


def als_formel(schaltung):
    if not isinstance(schaltung, tuple):
        return str(schaltung)
    zeichen, links, rechts = schaltung
    return '(' + als_formel(links) + ' ' + zeichen + ' ' + als_formel(rechts) + ')'

probe = ('|', ('+', 180, 820), 1000)
print(als_formel(probe), '=', schaltungswert(probe), 'Ω')
probe2 = ('|', ('+', ('|', 1200, 3300), 120), 1000)
print(als_formel(probe2), '=', round(schaltungswert(probe2), 3), 'Ω')
```

Das ist **genau dieselbe Rekursion** wie bei den Rechenausdrücken: nur andere Rechenregeln. Die zweite Schaltung ist die aus dem Aufgabenblatt: 1 200 Ω und 3 300 Ω parallel (etwa 880 Ω), dazu 120 Ω in Reihe (etwa 1 000 Ω), und das Ganze parallel zu 1 000 Ω, ergibt 500 Ω.

</details>

#### Aufgabe E.5

Die Funktion `alle_schaltungen(positionen)` ist vorgegeben. Sie bekommt ein Tupel von **Positionen** in der Kiste (so bleiben gleiche Werte auseinandergehalten) und gibt **alle** Schaltungen zurück, die genau diese Widerstände je einmal benutzen.
Das Prinzip: Ein einzelner Widerstand ist eine Schaltung. Sonst teilt man die Widerstände in eine **linke** und eine **rechte** Gruppe (der erste kommt immer nach links, damit man links und rechts nicht doppelt zählt),
baut für beide Gruppen rekursiv alle Schaltungen und setzt jede linke mit jeder rechten **in Reihe oder parallel** zusammen. Führe die Zelle aus.

**Erwartete Ausgabe:**
```
1 Widerstände: 1 Schaltungen
2 Widerstände: 2 Schaltungen
3 Widerstände: 12 Schaltungen
(100 + (220 + 180)) = 500
(100 | (220 + 180)) = 80.0
(100 + (220 | 180)) = 199.0
```

In [ ]:
def lies_widerstaende(dateiname):
    with open(dateiname, encoding='utf-8') as datei:
        werte = []
        for zeile in datei.read().split():
            werte.append(int(zeile))
    return werte


widerstaende = lies_widerstaende('daten/widerstaende.txt')

def schaltungswert(schaltung):
    if not isinstance(schaltung, tuple):             # ein einzelner Widerstand
        return schaltung
    zeichen, links, rechts = schaltung
    a = schaltungswert(links)
    b = schaltungswert(rechts)
    if zeichen == '+':
        return a + b
    return a * b / (a + b)


def als_formel(schaltung):
    if not isinstance(schaltung, tuple):
        return str(schaltung)
    zeichen, links, rechts = schaltung
    return '(' + als_formel(links) + ' ' + zeichen + ' ' + als_formel(rechts) + ')'

from itertools import combinations


def alle_schaltungen(positionen):
    """Alle Schaltungen, die genau die Widerstände an den Positionen (Tupel) der Kiste je einmal benutzen."""
    if len(positionen) == 1:
        return [widerstaende[positionen[0]]]
    erster = positionen[0]
    uebrige = positionen[1:]
    ergebnis = []
    for anzahl in range(0, len(uebrige)):            # so viele weitere Widerstände kommen nach links (rechts bleibt mindestens einer)
        for dazu in combinations(uebrige, anzahl):
            links = (erster,) + dazu
            rechts = []
            for p in uebrige:
                if p not in dazu:
                    rechts.append(p)
            for a in alle_schaltungen(links):
                for b in alle_schaltungen(tuple(rechts)):
                    ergebnis.append(('+', a, b))
                    ergebnis.append(('|', a, b))
    return ergebnis

for k in range(1, 4):
    print(k, 'Widerstände:', len(alle_schaltungen(tuple(range(k)))), 'Schaltungen')
for schaltung in alle_schaltungen((8, 10, 2))[:3]:
    print(als_formel(schaltung), '=', round(schaltungswert(schaltung), 3))

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
def lies_widerstaende(dateiname):
    with open(dateiname, encoding='utf-8') as datei:
        werte = []
        for zeile in datei.read().split():
            werte.append(int(zeile))
    return werte


widerstaende = lies_widerstaende('daten/widerstaende.txt')

def schaltungswert(schaltung):
    if not isinstance(schaltung, tuple):             # ein einzelner Widerstand
        return schaltung
    zeichen, links, rechts = schaltung
    a = schaltungswert(links)
    b = schaltungswert(rechts)
    if zeichen == '+':
        return a + b
    return a * b / (a + b)


def als_formel(schaltung):
    if not isinstance(schaltung, tuple):
        return str(schaltung)
    zeichen, links, rechts = schaltung
    return '(' + als_formel(links) + ' ' + zeichen + ' ' + als_formel(rechts) + ')'

from itertools import combinations


def alle_schaltungen(positionen):
    """Alle Schaltungen, die genau die Widerstände an den Positionen (Tupel) der Kiste je einmal benutzen."""
    if len(positionen) == 1:
        return [widerstaende[positionen[0]]]
    erster = positionen[0]
    uebrige = positionen[1:]
    ergebnis = []
    for anzahl in range(0, len(uebrige)):            # so viele weitere Widerstände kommen nach links (rechts bleibt mindestens einer)
        for dazu in combinations(uebrige, anzahl):
            links = (erster,) + dazu
            rechts = []
            for p in uebrige:
                if p not in dazu:
                    rechts.append(p)
            for a in alle_schaltungen(links):
                for b in alle_schaltungen(tuple(rechts)):
                    ergebnis.append(('+', a, b))
                    ergebnis.append(('|', a, b))
    return ergebnis

for k in range(1, 4):
    print(k, 'Widerstände:', len(alle_schaltungen(tuple(range(k)))), 'Schaltungen')
for schaltung in alle_schaltungen((8, 10, 2))[:3]:
    print(als_formel(schaltung), '=', round(schaltungswert(schaltung), 3))
```

Mit 1, 2 und 3 Widerständen gibt es 1, 2 und 12 Schaltungen. Bei drei Widerständen: Einen auswählen (3 Möglichkeiten), ihn mit einer der 2 Schaltungen der anderen beiden verbinden und das auf 2 Arten: 3 · 2 · 2 = 12.
Die Baumstruktur nutzt die Rekursion: Eine Schaltung aus zwei Gruppen ist ein Knoten, dessen Kinder Schaltungen aus den Gruppen sind. `alle_schaltungen` **erzeugt** Bäume, wo `schaltungswert` sie **auswertet**.

</details>

#### Aufgabe E.6

Schreibe `beste_schaltung(r, kmax)`: Sie probiert für jedes `k` von 1 bis `kmax` **alle** Auswahlen von `k` Positionen (`combinations(range(len(widerstaende)), k)`) und alle zugehörigen Schaltungen durch. Gib `(abstand, schaltung, geprueft)` zurück: den kleinsten Abstand `abs(Wert - r)`, die zugehörige Schaltung und die Zahl der geprüften Schaltungen.
Ein neuer Treffer zählt nur, wenn sein Abstand **kleiner als der bisher beste minus 0.000000001** ist (Toleranz gegen Rundungsfehler bei Kommazahlen).

**Erwartete Ausgabe:**
```
R = 500 : (1000 | (180 + 820)) = 500.0 Ω, Abstand 0.0
R = 140 : (180 | (820 | 2700)) = 139.949 Ω, Abstand 0.051
R = 315 : (1800 | (560 | 1200)) = 315.0 Ω, Abstand 0.0
R = 2719 : (1000 + (220 + 1500)) = 2720 Ω, Abstand 1
24864 Schaltungen geprüft
```

In [ ]:
def lies_widerstaende(dateiname):
    with open(dateiname, encoding='utf-8') as datei:
        werte = []
        for zeile in datei.read().split():
            werte.append(int(zeile))
    return werte


widerstaende = lies_widerstaende('daten/widerstaende.txt')

def schaltungswert(schaltung):
    if not isinstance(schaltung, tuple):             # ein einzelner Widerstand
        return schaltung
    zeichen, links, rechts = schaltung
    a = schaltungswert(links)
    b = schaltungswert(rechts)
    if zeichen == '+':
        return a + b
    return a * b / (a + b)


def als_formel(schaltung):
    if not isinstance(schaltung, tuple):
        return str(schaltung)
    zeichen, links, rechts = schaltung
    return '(' + als_formel(links) + ' ' + zeichen + ' ' + als_formel(rechts) + ')'

from itertools import combinations


def alle_schaltungen(positionen):
    """Alle Schaltungen, die genau die Widerstände an den Positionen (Tupel) der Kiste je einmal benutzen."""
    if len(positionen) == 1:
        return [widerstaende[positionen[0]]]
    erster = positionen[0]
    uebrige = positionen[1:]
    ergebnis = []
    for anzahl in range(0, len(uebrige)):            # so viele weitere Widerstände kommen nach links (rechts bleibt mindestens einer)
        for dazu in combinations(uebrige, anzahl):
            links = (erster,) + dazu
            rechts = []
            for p in uebrige:
                if p not in dazu:
                    rechts.append(p)
            for a in alle_schaltungen(links):
                for b in alle_schaltungen(tuple(rechts)):
                    ergebnis.append(('+', a, b))
                    ergebnis.append(('|', a, b))
    return ergebnis

def beste_schaltung(r, kmax):
    beste = (1e18, 0)               # (Abstand, Schaltung); die 0 ist nur ein Platzhalter
    geprueft = 0
    # dein Code
    return beste[0], beste[1], geprueft

for r in [500, 140, 315, 2719]:
    abstand, schaltung, geprueft = beste_schaltung(r, 3)
    print('R =', r, ':', als_formel(schaltung), '=', round(schaltungswert(schaltung), 3), 'Ω, Abstand', round(abstand, 3))
print(geprueft, 'Schaltungen geprüft')

<details>
<summary>Tipp anzeigen</summary>

Drei Schleifen ineinander: `for k in range(1, kmax + 1)`, `for positionen in combinations(range(len(widerstaende)), k)` und `for schaltung in alle_schaltungen(positionen)`. Darin zählst du `geprueft` hoch und vergleichst `abs(schaltungswert(schaltung) - r)` mit `beste[0]`.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
def lies_widerstaende(dateiname):
    with open(dateiname, encoding='utf-8') as datei:
        werte = []
        for zeile in datei.read().split():
            werte.append(int(zeile))
    return werte


widerstaende = lies_widerstaende('daten/widerstaende.txt')

def schaltungswert(schaltung):
    if not isinstance(schaltung, tuple):             # ein einzelner Widerstand
        return schaltung
    zeichen, links, rechts = schaltung
    a = schaltungswert(links)
    b = schaltungswert(rechts)
    if zeichen == '+':
        return a + b
    return a * b / (a + b)


def als_formel(schaltung):
    if not isinstance(schaltung, tuple):
        return str(schaltung)
    zeichen, links, rechts = schaltung
    return '(' + als_formel(links) + ' ' + zeichen + ' ' + als_formel(rechts) + ')'

from itertools import combinations


def alle_schaltungen(positionen):
    """Alle Schaltungen, die genau die Widerstände an den Positionen (Tupel) der Kiste je einmal benutzen."""
    if len(positionen) == 1:
        return [widerstaende[positionen[0]]]
    erster = positionen[0]
    uebrige = positionen[1:]
    ergebnis = []
    for anzahl in range(0, len(uebrige)):            # so viele weitere Widerstände kommen nach links (rechts bleibt mindestens einer)
        for dazu in combinations(uebrige, anzahl):
            links = (erster,) + dazu
            rechts = []
            for p in uebrige:
                if p not in dazu:
                    rechts.append(p)
            for a in alle_schaltungen(links):
                for b in alle_schaltungen(tuple(rechts)):
                    ergebnis.append(('+', a, b))
                    ergebnis.append(('|', a, b))
    return ergebnis

def beste_schaltung(r, kmax):
    """Beste Schaltung mit höchstens kmax Widerständen: (Abstand, Schaltung, Zahl der geprüften Schaltungen)."""
    beste = (1e18, None)
    geprueft = 0
    for k in range(1, kmax + 1):
        for positionen in combinations(range(len(widerstaende)), k):
            for schaltung in alle_schaltungen(positionen):
                geprueft = geprueft + 1
                abstand = abs(schaltungswert(schaltung) - r)
                if abstand < beste[0] - 0.000000001:     # nur echt besser (mit winziger Toleranz für Rundungsfehler)
                    beste = (abstand, schaltung)
    return beste[0], beste[1], geprueft

for r in [500, 140, 315, 2719]:
    abstand, schaltung, geprueft = beste_schaltung(r, 3)
    print('R =', r, ':', als_formel(schaltung), '=', round(schaltungswert(schaltung), 3), 'Ω, Abstand', round(abstand, 3))
print(geprueft, 'Schaltungen geprüft')
```

Für R = 500 gibt es mit drei Widerständen eine genaue Lösung (Abstand 0): `(1000 | (180 + 820))`, die Schaltung aus der Einleitung. Für 315 findet man ebenfalls eine genaue (in den Lösungsnotebooks stehen die Ergebnisse für k = 4 und alle Wunschwerte).
Bei 140 und 2719 reichen drei Widerstände nicht für einen genauen Wert, der Abstand bleibt klein (0,051 beziehungsweise 1). Geprüft werden 24 + 276 · 2 + 2 024 · 12 = **24 864** Schaltungen. Für vier Widerstände wären es schon über 1,2 Millionen, jeweils mit einem Baum,
den man zuerst bauen und dann auswerten muss: Dort lohnt sich ein Trick (siehe Lösungsnotebook).

</details>

#### Frage E.7

1. Warum ist `schaltungswert` hier genauso gebaut wie `werte_aus` aus Abschnitt 1? Was ist anders, was gleich?
2. Mit 24 Widerständen gibt es für k = 3 nur 24 864 Schaltungen, für k = 4 aber über eine Million. Wie wächst die Zahl der Möglichkeiten, wenn die Kiste 100 statt 24 Widerstände enthält? (Überschlage mit der Zahl der Auswahlen von 4 aus 100.)

✏️ *Deine Antwort (Doppelklick zum Bearbeiten):*

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

1. **Gleich:** Basisfall für das Blatt (eine Zahl), sonst zerlegen, beide Teile rekursiv auswerten und je nach Zeichen verknüpfen. **Anders:** nur die Rechenregel (`+` und `*` beziehungsweise `+` und `a * b / (a + b)`) und das Ergebnis (Kommazahl statt ganzer Zahl). Wer den Baum-Auswerter einmal hat, kann ihn für viele Aufgaben abwandeln.
2. Die Zahl der Auswahlen von 4 aus 100 Widerständen ist `math.comb(100, 4)` = 3 921 225, und jede Auswahl hat noch 120 Schaltungen: knapp **470 Millionen** Schaltungen. Das ist zu viel zum einfachen Durchprobieren. Deshalb braucht man bei größeren Kisten einen geschickteren Weg (zum Beispiel drei Widerstände auswählen und den vierten ausrechnen).

</details>

## Wettbewerbs-Ecke für Schnelle: Drehfreudig

*Angelehnt an Aufgabe 1 „Drehfreudig?“ der 1. Runde des 44. Bundeswettbewerbs Informatik (2025/26).*

Ein Baum wird als Klammerfolge angegeben, zum Beispiel `(()())`. Man zeichnet ihn mit **Rechtecken**: Die Wurzel ist ein Rechteck der Breite 1. Die Rechtecke der Kinder eines Knotens haben alle dieselbe Breite
und liegen nebeneinander genau unter dem Rechteck des Knotens. Hat die Wurzel zwei Kinder, ist jedes 1/2 breit, hat eines von ihnen wieder drei Kinder, sind diese 1/6 breit.

Dreht man das Bild um 180° und legt es unter das Original, sollen sich die **Blätter** in der Mitte genau treffen. Dann heißt der Baum **drehfreudig**. Dafür muss die Folge der **Breiten der Blätter** von links nach rechts
**dieselbe** sein wie von rechts nach links (ein Palindrom). Wir vereinfachen: Wir prüfen nur diese Bedingung an den Breiten (die Höhen der Rechtecke lassen wir außer Acht, so wie es auch die Aufgabe tut).
Die Beispiele sind `daten/drehfreudig01.txt` bis `drehfreudig07.txt`, jeweils eine Klammerfolge.

#### Aufgabe E.8

Die Funktionen `klammern_zu_baum` (aus 4.4) und `lies_klammern` sind vorgegeben. Schreibe

- `blatt_nenner(baum, nenner=1)`: Eine Liste der **Nenner** der Blattbreiten von links nach rechts (die Breite eines Blatts ist `1 / nenner`; wir rechnen mit dem Nenner, um Brüche zu vermeiden). Ein Blatt (leere Liste) gibt `[nenner]` zurück. Ein Knoten mit `m` Kindern gibt an jedes Kind den Nenner `nenner * m` weiter, und die Ergebnislisten der Kinder werden **hintereinandergehängt**.
- `ist_drehfreudig(text)`: `True`, wenn die Nenner-Liste gleich ihrer umgekehrten Liste (`liste[::-1]`) ist.

**Erwartete Ausgabe:**
```
[4, 4, 6, 6, 6]
drehfreudig01: True
drehfreudig02: False
drehfreudig03: False
drehfreudig04: True
drehfreudig05: False
drehfreudig06: True
drehfreudig07: False
```

In [ ]:
def klammern_zu_baum(text):
    stapel = [[]]                               # ein Hilfsknoten ganz unten
    for zeichen in text:
        if zeichen == '(':
            stapel.append([])                   # neuer Knoten, seine Klammer ist offen
        else:
            knoten = stapel.pop()               # fertig ...
            stapel[-1].append(knoten)           # ... und als Kind an den Knoten darunter hängen
    return stapel[0][0]                         # die Wurzel

def lies_klammern(dateiname):
    with open(dateiname, encoding='utf-8') as datei:
        return datei.read().strip()

def blatt_nenner(baum, nenner=1):
    # dein Code
    return []


def ist_drehfreudig(text):
    # dein Code
    return False

print(blatt_nenner(klammern_zu_baum('((()())(()()()))')))
for nr in range(1, 8):
    text = lies_klammern('daten/drehfreudig0' + str(nr) + '.txt')
    print('drehfreudig0' + str(nr) + ':', ist_drehfreudig(text))

<details>
<summary>Tipp anzeigen</summary>

Basisfall: `if len(baum) == 0: return [nenner]`. Sonst `ergebnis = []` und für jedes Kind `ergebnis = ergebnis + blatt_nenner(kind, nenner * len(baum))`. Bei `ist_drehfreudig` bildest du aus `text` zuerst mit `klammern_zu_baum` den Baum.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
def klammern_zu_baum(text):
    stapel = [[]]                               # ein Hilfsknoten ganz unten
    for zeichen in text:
        if zeichen == '(':
            stapel.append([])                   # neuer Knoten, seine Klammer ist offen
        else:
            knoten = stapel.pop()               # fertig ...
            stapel[-1].append(knoten)           # ... und als Kind an den Knoten darunter hängen
    return stapel[0][0]                         # die Wurzel

def lies_klammern(dateiname):
    with open(dateiname, encoding='utf-8') as datei:
        return datei.read().strip()

def blatt_nenner(baum, nenner=1):
    """Nenner der Breiten aller Blätter von links nach rechts (Breite = 1 / nenner)."""
    if len(baum) == 0:
        return [nenner]                              # Blatt
    ergebnis = []
    for kind in baum:
        ergebnis = ergebnis + blatt_nenner(kind, nenner * len(baum))
    return ergebnis


def ist_drehfreudig(text):
    nenner = blatt_nenner(klammern_zu_baum(text))
    return nenner == nenner[::-1]

print(blatt_nenner(klammern_zu_baum('((()())(()()()))')))
for nr in range(1, 8):
    text = lies_klammern('daten/drehfreudig0' + str(nr) + '.txt')
    print('drehfreudig0' + str(nr) + ':', ist_drehfreudig(text))
```

Der erste Baum `((()())(()()()))` ist der nicht drehfreudige Baum vom Aufgabenblatt: Die Blattbreiten sind 1/4, 1/4, 1/6, 1/6, 1/6, rückwärts 1/6, 1/6, 1/6, 1/4, 1/4, also kein Palindrom.
Drehfreudig sind unter den sieben Beispielen die Nummern 1, 4 und 6. Die Breiten sind **Brüche mit Zähler 1**, deshalb genügt der Nenner (zwei Breiten sind genau dann gleich, wenn ihre Nenner gleich sind), und man vermeidet Rundungsfehler bei Kommazahlen.

</details>

#### Frage E.9

1. Warum reicht es, den Baum **einmal** von links nach rechts durchzugehen, und wie viele Schritte braucht `ist_drehfreudig` bei einer Klammerfolge mit n Zeichen (O-Notation)?
2. Die Funktion `blatt_nenner` hängt Listen mit `ergebnis = ergebnis + ...` zusammen. Warum ist das hier (bei Bäumen mit einigen Tausend Knoten) unproblematisch? Was wäre eine schnellere Alternative?
3. Die Wurzel hat nur **ein** Kind (`((()()))`). Ändert das etwas an der Antwort, ob der Baum drehfreudig ist? Vergleiche mit `(()())`.

✏️ *Deine Antwort (Doppelklick zum Bearbeiten):*

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

1. Jeder Knoten wird genau einmal besucht (`klammern_zu_baum` einmal pro Zeichen, `blatt_nenner` einmal pro Knoten), und der Palindrom-Vergleich geht einmal über die Blätter. Das sind insgesamt **O(n)** Schritte.
2. Jedes Zusammenhängen kopiert die Listen. Das kann bei sehr vielen Blättern langsam werden, bei einigen Tausend Knoten ist es kein Problem, denn jedes Blatt wird nur so oft kopiert, wie es Ebenen über ihm gibt (höchstens die Höhe). Schneller geht es, wenn man **eine einzige** Ergebnisliste von außen mitgibt und an sie mit `append` anfügt.
3. Hat ein Knoten nur ein Kind, ändert sich der Nenner nicht (`nenner * 1`): Dieser Knoten verändert die Breiten nicht, man kann ihn bei der Prüfung der Breiten einfach übergehen. `((()()))` hat dieselben Blattbreiten wie `(()())`, nämlich zwei Blätter mit 1/2.

</details>

## Das hast du gelernt

```python
# Baum als verschachteltes Tupel (Rechenausdruck): Blatt = Zahl
def werte_aus(baum):
    if not isinstance(baum, tuple):       # Basisfall
        return baum
    zeichen, links, rechts = baum
    a = werte_aus(links)
    b = werte_aus(rechts)
    ...                                   # a und b verknüpfen

# Baum mit beliebig vielen Kindern: Knoten = Liste der Kinder, Blatt = []
def hoehe_baum(baum):
    hoechste = 0
    for kind in baum:
        hoechste = max(hoechste, hoehe_baum(kind))
    return 1 + hoechste

# Klammerfolge → Baum mit Stapel
stapel = [[]]
for zeichen in text:
    if zeichen == '(':
        stapel.append([])
    else:
        knoten = stapel.pop()
        stapel[-1].append(knoten)
baum = stapel[0][0]
```

- Ein **Baum** ist ein Graph ohne Kreis mit einer **Wurzel**; ein Baum mit n Knoten hat n − 1 Kanten. Jeder **Teilbaum** ist wieder ein Baum.
- Man speichert Bäume als **verschachtelte Tupel oder Listen**. Die Rekursion folgt der Struktur: **Basisfall** für das Blatt, sonst Teilbäume bearbeiten und Ergebnisse zusammensetzen. Testen mit dem kleinsten Baum und mit Knoten ohne Kinder.
- Summe, Zahl der Blätter, Höhe, Ausgabe als eingerückter Text oder als Klammerfolge: alles in **O(Knoten)**, weil jeder Knoten einmal drankommt. Ruft man pro Knoten den ganzen Teilbaum mehrfach auf, kann es viel mehr werden.
- **Klammerfolge prüfen:** Zähler, der nie unter 0 fallen darf und am Ende 0 sein muss. Die Zahl der `(` und `)` allein genügt nicht. **Klammerfolge zu Baum:** Stapel der offenen Knoten.
- **Rekursionsgrenze** (etwa 1 000): Bei sehr tiefen Bäumen nimmt man einen **Stapel** statt der Rekursion.
- Auch Schaltungen, Mobiles und Dateiordner sind Bäume. Dasselbe Muster wertet sie aus.

### Aufgaben der 1. Runde dazu

Bearbeite eine Aufgabe zuerst selbst und sieh dir das Lösungsnotebook erst danach an.

- [Mobile](https://ktheu.github.io/bwinfN/notebooks/index.html?path=bwinf1/33_Mobile/mobile_loesung.ipynb) (33. BwInf): ein balanciertes Mobile aus Gewichten bauen und prüfen
- [Widerstand](https://ktheu.github.io/bwinfN/notebooks/index.html?path=bwinf1/37_Widerstand/widerstand_loesung.ipynb) (37. BwInf): Schaltungen als Bäume, beste Näherung an einen Wunschwert
- [Drehfreudig](https://ktheu.github.io/bwinfN/notebooks/index.html?path=bwinf1/44_Drehfreudig/drehfreudig_loesung.ipynb) (44. BwInf): Klammerfolge als Baum, Palindrom der Blattbreiten

Mit diesem Notebook ist der **Kern des Lehrgangs (01 bis 13)** vollständig: Durchprobieren, Backtracking, Schranken, logisches Schließen, Breitensuche, Zustände, Dijkstra, Reihenfolgen, Bäume, Merken, Tabellen, Greedy und binäre Suche.
Als Nächstes lohnt sich der Blick in die Lösungsnotebooks zu den Aufgaben der 1. Runde (Übersicht in `bwinf1.md`): Dort siehst du, wie diese Werkzeuge in echten Aufgaben zusammenspielen, und die Schritte, die du hier eingeübt hast, erkennst du schnell wieder.